In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import pyproj as proj

from flux_footprint.utils import find_transform, mask_fp_cutoff
from flux_footprint import calc_footprint_FFP as fp_1
from flux_footprint import calc_footprint_FFP_climatology as fp_2
print("Kljun FFP model imported successfully!")

dir_output = r"C:\GitHub\flux-footprint-py\outputs"
if not os.path.exists(dir_output):
    os.makedirs(dir_output)
    print(f"{dir_output} is created successfully.")
else:
    print(f"{dir_output} already exists.")

Kljun FFP model imported successfully!
C:\GitHub\flux-footprint-py\outputs already exists.


In [3]:
file_path = r'C:\GitHub\flux-footprint-py\example_data\example_heavy.txt'
df = pd.read_csv(file_path, sep='\t') 

# Retrieve data from the DataFrame
zm = df['zm'].tolist()                  # Measurement height (m)
hc = df['hc'].tolist()                  # Canopy height (m)
z0 = df['z0'].tolist()                  # Roughness length (m)
umean = df['umean'].tolist()            # Mean wind speed (m/s)
h = df['h'].tolist()                    # Boundary layer height (m)
ol = df['ol'].tolist()                  # Obukhov length (m)
sigmav = df['sigmav'].tolist()          # Standard deviation of vertical wind speed (m/s)
ustar = df['ustar'].tolist()            # Friction velocity (m/s)
wind_dir = df['wind_dir'].tolist()      # Wind direction (degrees)

# Set up parameters for footprint calculation
domain = None
dx, dy = 3, 3
nx, ny = 200, 200
rs = [0.9]      
crop = 0
fig = 1

# Site coordinates
latitude = 38.1235	
longitude = -121.549
cutoff = 0.9

filename = "US-xSP_footprint_test_delete_heavy.tif"

In [ ]:
FFP_2 = fp_2.FFP_climatology(zm=zm,
                             z0=z0,
                             umean=umean,
                             h=h,
                             ol=ol,
                             sigmav=sigmav,
                             ustar=ustar,
                             wind_dir=wind_dir,
                             dx=dx,
                             dy=dy,
                             rs=rs,
                             crop=crop,
                             fig=0)

# Save results as GeoTIFF
# convert locations into UTM Zone 10N.
cor_x = round(longitude,4)
cor_y = round(latitude,4)
site_cord = (cor_x,cor_y) 
# WGS 84 Geographic Coordinate System
prj_gcs = proj.Proj(init='EPSG:4326')
# WGS 84 / UTM Zone 10N Projected Coordinate System
prj_pcs = proj.Proj(init='EPSG:32610')
(site_x,site_y) = proj.transform(prj_gcs,prj_pcs,*site_cord)

output_data = None
f_2d = np.array(FFP_2['fclim_2d'])
x_2d = np.array(FFP_2['x_2d']) + site_x
y_2d = np.array(FFP_2['y_2d']) + site_y
f_2d = f_2d*dx**2

#Calculate affine transform for given x_2d and y_2d
affine_transform = find_transform(x_2d,y_2d)
file_output_path = os.path.join(dir_output, filename)
new_dat = rasterio.open(file_output_path,'w',driver='GTiff',dtype=rasterio.float64,
                        count=1,height=f_2d.shape[0],width=f_2d.shape[1],
                        transform=affine_transform,
                        crs='+proj=utm +zone=10 +ellps=WGS84 +datum=WGS84 +units=m +no_defs',
                        nodata=0.00000000e+000)
# #Mask out points that are below a % threshold (defaults to 90%)
f_2d = mask_fp_cutoff(f_2d, cutoff=cutoff)
#Write the new band
new_dat.write(f_2d,1)
new_dat.close()
new_dat = None


Alert(0013):
 Using z0, ignoring umean if passed.
 Execution continues.

Calculating footprint  1  of  75552

Error(0008):
 sigmav (standard deviation of crosswind) must be larger than zero.
 Execution continues.

Error(0016):
 At least one required input is invalid. Skipping current footprint.
 Execution continues.

Error(0008):
 sigmav (standard deviation of crosswind) must be larger than zero.
 Execution continues.

Error(0016):
 At least one required input is invalid. Skipping current footprint.
 Execution continues.

Error(0008):
 sigmav (standard deviation of crosswind) must be larger than zero.
 Execution continues.

Error(0016):
 At least one required input is invalid. Skipping current footprint.
 Execution continues.

Error(0008):
 sigmav (standard deviation of crosswind) must be larger than zero.
 Execution continues.

Error(0016):
 At least one required input is invalid. Skipping current footprint.
 Execution continues.

Error(0008):
 sigmav (standard deviation of crosswind